# ICE COT Report — Petroleum Complex Overview

Basic interrogation of the ICE Europe Commitments of Traders report
for petroleum products (Brent, Gasoil).

This data covers ICE Futures Europe contracts, which are **not** in
the CFTC disaggregated report.

In [22]:
import pandas as pd
import numpy as np

ICE_PATH = '/Users/oualid/Documents/Projects/omroot_repos/cot-ingest/downloads/ice/ice_cot.csv'
raw = pd.read_csv(ICE_PATH, low_memory=False)

print(f'Shape: {raw.shape}')
print(f'Columns: {raw.shape[1]}')
raw['date'] = pd.to_datetime(raw['As_of_Date_Form_MM/DD/YYYY'])
print(f'Date range: {raw["date"].min().date()} -> {raw["date"].max().date()}')
print(f'Unique markets: {raw["Market_and_Exchange_Names"].nunique()}')

Shape: (8149, 198)
Columns: 198
Date range: 2011-01-04 -> 2026-03-24
Unique markets: 13


## 0. Column inventory

The ICE COT report has 198 columns. Most follow a pattern:
`{TraderCategory}_Positions_{Side}_{Scope}` where:
- **Side**: Long, Short, Spread
- **Scope**: `All` (all contracts), `Old` (old crop / front months), `Other` (back months / other)

Columns are grouped below by purpose.

## 1. All markets in the report

In [23]:
raw['Open_Interest_All'] = pd.to_numeric(raw['Open_Interest_All'], errors='coerce')

markets = raw.groupby('Market_and_Exchange_Names').agg(
    rows=('date', 'count'),
    start=('date', 'min'),
    end=('date', 'max'),
    avg_oi=('Open_Interest_All', 'mean'),
    last_oi=('Open_Interest_All', 'last'),
).sort_values('rows', ascending=False)

print(f'All markets ({len(markets)}):')
for name, r in markets.iterrows():
    print(f'  {name:<60s}  rows={r["rows"]:>5d}  {r["start"].date()} -> {r["end"].date()}'
          f'  avg_OI={r["avg_oi"]:>12,.0f}  last_OI={r["last_oi"]:>12,.0f}')

All markets (13):
  ICE Brent Crude Futures - ICE Futures Europe                  rows=  909  2011-01-04 -> 2026-03-24  avg_OI=   1,947,967  last_OI=   3,184,135
  ICE Gasoil Futures - ICE Futures Europe                       rows=  909  2011-01-04 -> 2026-03-24  avg_OI=     741,848  last_OI=     902,564
  ICE Brent Crude Futures and Options - ICE Futures Europe      rows=  681  2013-03-12 -> 2026-03-24  avg_OI=   2,824,978  last_OI=   4,910,557
  ICE Gasoil Futures and Options - ICE Futures Europe           rows=  681  2013-03-12 -> 2026-03-24  avg_OI=     821,798  last_OI=   1,053,523
  ICE Cocoa Futures - ICE Futures Europe                        rows=  600  2014-09-30 -> 2026-03-24  avg_OI=     257,896  last_OI=     206,068
  ICE Cocoa Futures and Options - ICE Futures Europe            rows=  600  2014-09-30 -> 2026-03-24  avg_OI=     343,098  last_OI=     278,276
  ICE Robusta Coffee Futures - ICE Futures Europe               rows=  600  2014-09-30 -> 2026-03-24  avg_OI=     110,

## 2. Petroleum contracts

In [24]:
# Filter for petroleum-related contracts
PETRO_KEYWORDS = ['Brent', 'Gasoil', 'Heating', 'Gas Oil', 'Diesel']

petro_mask = raw['Market_and_Exchange_Names'].str.contains(
    '|'.join(PETRO_KEYWORDS), case=False, na=False
)
petro = raw[petro_mask].copy()

print(f'Petroleum rows: {len(petro)} ({len(petro)/len(raw)*100:.1f}% of total)')
print()

petro_markets = petro.groupby('Market_and_Exchange_Names').agg(
    rows=('date', 'count'),
    start=('date', 'min'),
    end=('date', 'max'),
).sort_values('rows', ascending=False)

print('Petroleum contracts:')
for name, r in petro_markets.iterrows():
    print(f'  {name:<60s}  rows={r["rows"]:>5d}  {r["start"].date()} -> {r["end"].date()}')

Petroleum rows: 3180 (39.0% of total)

Petroleum contracts:
  ICE Brent Crude Futures - ICE Futures Europe                  rows=  909  2011-01-04 -> 2026-03-24
  ICE Gasoil Futures - ICE Futures Europe                       rows=  909  2011-01-04 -> 2026-03-24
  ICE Brent Crude Futures and Options - ICE Futures Europe      rows=  681  2013-03-12 -> 2026-03-24
  ICE Gasoil Futures and Options - ICE Futures Europe           rows=  681  2013-03-12 -> 2026-03-24


## 3. Futures-only vs Combined (Futures + Options)

In [25]:
print('FutOnly_or_Combined values:')
print(raw['FutOnly_or_Combined'].value_counts().to_string())

# For Brent specifically
for name in ['ICE Brent Crude Futures - ICE Futures Europe',
             'ICE Brent Crude Futures and Options - ICE Futures Europe']:
    sub = raw[raw['Market_and_Exchange_Names'] == name]
    if len(sub) > 0:
        print(f'\n{name}:')
        print(f'  Rows: {len(sub)}')
        print(f'  Range: {sub["date"].min().date()} -> {sub["date"].max().date()}')
        print(f'  Type: {sub["FutOnly_or_Combined"].unique()}')

FutOnly_or_Combined values:
FutOnly_or_Combined
FutOnly     4159
Combined    3990

ICE Brent Crude Futures - ICE Futures Europe:
  Rows: 909
  Range: 2011-01-04 -> 2026-03-24
  Type: ['FutOnly' 'Combined']

ICE Brent Crude Futures and Options - ICE Futures Europe:
  Rows: 681
  Range: 2013-03-12 -> 2026-03-24
  Type: ['Combined']


## 4. Column structure

ICE uses CamelCase vs CFTC's snake_case, but the same 5 categories.

In [26]:
CATEGORIES = {
    'Producer/Merchant': {
        'long': 'Prod_Merc_Positions_Long_All',
        'short': 'Prod_Merc_Positions_Short_All',
    },
    'Swap Dealers': {
        'long': 'Swap_Positions_Long_All',
        'short': 'Swap__Positions_Short_All',  # double underscore
        'spread': 'Swap__Positions_Spread_All',
    },
    'Managed Money': {
        'long': 'M_Money_Positions_Long_All',
        'short': 'M_Money_Positions_Short_All',
        'spread': 'M_Money_Positions_Spread_All',
    },
    'Other Reportables': {
        'long': 'Other_Rept_Positions_Long_All',
        'short': 'Other_Rept_Positions_Short_All',
        'spread': 'Other_Rept_Positions_Spread_All',
    },
    'Non-Reportables': {
        'long': 'NonRept_Positions_Long_All',
        'short': 'NonRept_Positions_Short_All',
    },
}

print('Trader category column mapping (ICE):')
print()
for cat, cols_map in CATEGORIES.items():
    print(f'{cat}:')
    for side, col in cols_map.items():
        exists = col in raw.columns
        print(f'  {side:8s} -> {col}  {"OK" if exists else "MISSING!"}')
    print()

# Note: ICE also has duplicate Swap columns without double underscore
swap_dupes = [c for c in raw.columns if 'Swap' in c and 'Positions' in c]
print('All Swap position columns:')
for c in swap_dupes:
    print(f'  {c}')

Trader category column mapping (ICE):

Producer/Merchant:
  long     -> Prod_Merc_Positions_Long_All  OK
  short    -> Prod_Merc_Positions_Short_All  OK

Swap Dealers:
  long     -> Swap_Positions_Long_All  OK
  short    -> Swap__Positions_Short_All  OK
  spread   -> Swap__Positions_Spread_All  OK

Managed Money:
  long     -> M_Money_Positions_Long_All  OK
  short    -> M_Money_Positions_Short_All  OK
  spread   -> M_Money_Positions_Spread_All  OK

Other Reportables:
  long     -> Other_Rept_Positions_Long_All  OK
  short    -> Other_Rept_Positions_Short_All  OK
  spread   -> Other_Rept_Positions_Spread_All  OK

Non-Reportables:
  long     -> NonRept_Positions_Long_All  OK
  short    -> NonRept_Positions_Short_All  OK

All Swap position columns:
  Swap_Positions_Long_All
  Swap__Positions_Short_All
  Swap__Positions_Spread_All
  Swap_Positions_Long_Old
  Swap__Positions_Short_Old
  Swap__Positions_Spread_Old
  Swap_Positions_Long_Other
  Swap__Positions_Short_Other
  Swap__Positions_S

## 5. Primary petroleum contracts

In [27]:
PRIMARY = {
    'Brent (F+O)': 'ICE Brent Crude Futures and Options - ICE Futures Europe',
    'Brent (F)':   'ICE Brent Crude Futures - ICE Futures Europe',
    'Gasoil (F+O)':'ICE Gasoil Futures and Options - ICE Futures Europe',
    'Gasoil (F)':  'ICE Gasoil Futures - ICE Futures Europe',
}

print(f'{"Ticker":<16s}  {"Rows":>5s}  {"Type":<10s}  Date range')
print('-' * 70)
for ticker, name in PRIMARY.items():
    sub = raw[raw['Market_and_Exchange_Names'] == name]
    if len(sub) > 0:
        ftype = sub['FutOnly_or_Combined'].iloc[0]
        print(f'{ticker:<16s}  {len(sub):>5d}  {ftype:<10s}  '
              f'{sub["date"].min().date()} -> {sub["date"].max().date()}')
    else:
        print(f'{ticker:<16s}  NOT FOUND')

print()
print('Note: Combined (F+O) available from 2013-03-12.')
print('Use Futures-only as fallback for earlier dates.')

Ticker             Rows  Type        Date range
----------------------------------------------------------------------
Brent (F+O)         681  Combined    2013-03-12 -> 2026-03-24
Brent (F)           909  FutOnly     2011-01-04 -> 2026-03-24
Gasoil (F+O)        681  Combined    2013-03-12 -> 2026-03-24
Gasoil (F)          909  FutOnly     2011-01-04 -> 2026-03-24

Note: Combined (F+O) available from 2013-03-12.
Use Futures-only as fallback for earlier dates.


## 6. Data quality checks

In [28]:
for ticker, name in PRIMARY.items():
    sub = raw[raw['Market_and_Exchange_Names'] == name].copy()
    if len(sub) == 0:
        continue
    
    print(f'=== {ticker} ===')
    
    # Missing key columns
    for c in ['Open_Interest_All', 'M_Money_Positions_Long_All']:
        n_miss = sub[c].isna().sum()
        if n_miss > 0:
            print(f'  Missing {c}: {n_miss}')
    
    # Duplicates
    n_dup = sub['date'].duplicated().sum()
    print(f'  Duplicate dates: {n_dup}')
    
    # Zero OI
    oi = pd.to_numeric(sub['Open_Interest_All'], errors='coerce')
    print(f'  Zero OI weeks: {(oi == 0).sum()}')
    
    # Gaps
    dates_sorted = sub['date'].sort_values()
    diffs = dates_sorted.diff().dt.days
    big_gaps = diffs[diffs > 10]
    print(f'  Date gaps > 10 days: {len(big_gaps)}')
    if len(big_gaps) > 0:
        for idx in big_gaps.index[:3]:
            prev = dates_sorted.iloc[dates_sorted.index.get_loc(idx)-1]
            print(f'    {prev.date()} -> {dates_sorted.loc[idx].date()} ({int(diffs.loc[idx])}d)')
    
    # Day of week
    dow = sub['date'].dt.day_name().value_counts()
    print(f'  Day-of-week: {dow.to_dict()}')
    print()

=== Brent (F+O) ===
  Duplicate dates: 0
  Zero OI weeks: 0
  Date gaps > 10 days: 0
  Day-of-week: {'Tuesday': 679, 'Monday': 2}

=== Brent (F) ===
  Duplicate dates: 114
  Zero OI weeks: 0
  Date gaps > 10 days: 0
  Day-of-week: {'Tuesday': 903, 'Monday': 6}

=== Gasoil (F+O) ===
  Duplicate dates: 0
  Zero OI weeks: 0
  Date gaps > 10 days: 0
  Day-of-week: {'Tuesday': 679, 'Monday': 2}

=== Gasoil (F) ===
  Duplicate dates: 114
  Zero OI weeks: 0
  Date gaps > 10 days: 0
  Day-of-week: {'Tuesday': 903, 'Monday': 6}



## 7. Position identity check — Brent Combined

In [29]:
br = raw[raw['Market_and_Exchange_Names'] ==
         'ICE Brent Crude Futures and Options - ICE Futures Europe'].copy()

num_cols = ['Open_Interest_All',
            'Prod_Merc_Positions_Long_All', 'Prod_Merc_Positions_Short_All',
            'Swap_Positions_Long_All', 'Swap__Positions_Short_All', 'Swap__Positions_Spread_All',
            'M_Money_Positions_Long_All', 'M_Money_Positions_Short_All', 'M_Money_Positions_Spread_All',
            'Other_Rept_Positions_Long_All', 'Other_Rept_Positions_Short_All', 'Other_Rept_Positions_Spread_All',
            'Tot_Rept_Positions_Long_All', 'Tot_Rept_Positions_Short_All',
            'NonRept_Positions_Long_All', 'NonRept_Positions_Short_All']
for c in num_cols:
    br[c] = pd.to_numeric(br[c], errors='coerce')

# Reportable long = PM_long + SD_long + SD_spread + MM_long + MM_spread + OR_long + OR_spread
br['sum_rept_long'] = (br['Prod_Merc_Positions_Long_All']
    + br['Swap_Positions_Long_All'] + br['Swap__Positions_Spread_All']
    + br['M_Money_Positions_Long_All'] + br['M_Money_Positions_Spread_All']
    + br['Other_Rept_Positions_Long_All'] + br['Other_Rept_Positions_Spread_All'])

br['sum_rept_short'] = (br['Prod_Merc_Positions_Short_All']
    + br['Swap__Positions_Short_All'] + br['Swap__Positions_Spread_All']
    + br['M_Money_Positions_Short_All'] + br['M_Money_Positions_Spread_All']
    + br['Other_Rept_Positions_Short_All'] + br['Other_Rept_Positions_Spread_All'])

diff_long = (br['sum_rept_long'] - br['Tot_Rept_Positions_Long_All']).abs()
diff_short = (br['sum_rept_short'] - br['Tot_Rept_Positions_Short_All']).abs()

print('Brent Combined position identity check:')
print(f'  Rept long  — max diff: {diff_long.max():.0f}, mean: {diff_long.mean():.1f}')
print(f'  Rept short — max diff: {diff_short.max():.0f}, mean: {diff_short.mean():.1f}')

# OI = reportable + non-reportable
diff_oi_l = (br['Tot_Rept_Positions_Long_All'] + br['NonRept_Positions_Long_All'] - br['Open_Interest_All']).abs()
diff_oi_s = (br['Tot_Rept_Positions_Short_All'] + br['NonRept_Positions_Short_All'] - br['Open_Interest_All']).abs()
print(f'  OI = rept_long + nonrept_long  — max diff: {diff_oi_l.max():.0f}')
print(f'  OI = rept_short + nonrept_short — max diff: {diff_oi_s.max():.0f}')

Brent Combined position identity check:
  Rept long  — max diff: nan, mean: nan
  Rept short — max diff: nan, mean: nan
  OI = rept_long + nonrept_long  — max diff: nan
  OI = rept_short + nonrept_short — max diff: nan


## 8. Sample data: Brent Combined

In [30]:
sample_cols = [
    'As_of_Date_Form_MM/DD/YYYY',
    'Open_Interest_All',
    'Prod_Merc_Positions_Long_All', 'Prod_Merc_Positions_Short_All',
    'Swap_Positions_Long_All', 'Swap__Positions_Short_All',
    'M_Money_Positions_Long_All', 'M_Money_Positions_Short_All',
    'Other_Rept_Positions_Long_All', 'Other_Rept_Positions_Short_All',
    'NonRept_Positions_Long_All', 'NonRept_Positions_Short_All',
]

print('Last 5 weeks of Brent Combined:')
print(br[sample_cols].tail(5).to_string(index=False))

Last 5 weeks of Brent Combined:
As_of_Date_Form_MM/DD/YYYY  Open_Interest_All  Prod_Merc_Positions_Long_All  Prod_Merc_Positions_Short_All  Swap_Positions_Long_All  Swap__Positions_Short_All  M_Money_Positions_Long_All  M_Money_Positions_Short_All  Other_Rept_Positions_Long_All  Other_Rept_Positions_Short_All  NonRept_Positions_Long_All  NonRept_Positions_Short_All
                2026-02-24          4179180.0                       1322924                        1727763                   438375                        NaN                      413137                        92185                         196145                          499823                       73280                        55006
                2026-03-03          4186136.0                       1287565                        1683000                   413871                        NaN                      360987                        75393                         191464                          486141                  

## 9. CFTC vs ICE column name comparison

Quick reference for translating between the two reports.

In [31]:
mapping = [
    ('Date', 'report_date_as_yyyy_mm_dd', 'As_of_Date_Form_MM/DD/YYYY'),
    ('Market', 'contract_market_name', 'Market_and_Exchange_Names'),
    ('Code', 'cftc_contract_market_code', 'CFTC_Contract_Market_Code'),
    ('OI', 'open_interest_all', 'Open_Interest_All'),
    ('PM Long', 'prod_merc_positions_long', 'Prod_Merc_Positions_Long_All'),
    ('PM Short', 'prod_merc_positions_short', 'Prod_Merc_Positions_Short_All'),
    ('SD Long', 'swap_positions_long_all', 'Swap_Positions_Long_All'),
    ('SD Short', 'swap__positions_short_all', 'Swap__Positions_Short_All'),
    ('MM Long', 'm_money_positions_long_all', 'M_Money_Positions_Long_All'),
    ('MM Short', 'm_money_positions_short_all', 'M_Money_Positions_Short_All'),
    ('OR Long', 'other_rept_positions_long', 'Other_Rept_Positions_Long_All'),
    ('OR Short', 'other_rept_positions_short', 'Other_Rept_Positions_Short_All'),
    ('NR Long', 'nonrept_positions_long_all', 'NonRept_Positions_Long_All'),
    ('NR Short', 'nonrept_positions_short_all', 'NonRept_Positions_Short_All'),
    ('Type', 'futonly_or_combined', 'FutOnly_or_Combined'),
]

print(f'{"Field":<10s}  {"CFTC (snake_case)":<35s}  {"ICE (CamelCase)"}')
print('-' * 90)
for field, cftc, ice in mapping:
    print(f'{field:<10s}  {cftc:<35s}  {ice}')

Field       CFTC (snake_case)                    ICE (CamelCase)
------------------------------------------------------------------------------------------
Date        report_date_as_yyyy_mm_dd            As_of_Date_Form_MM/DD/YYYY
Market      contract_market_name                 Market_and_Exchange_Names
Code        cftc_contract_market_code            CFTC_Contract_Market_Code
OI          open_interest_all                    Open_Interest_All
PM Long     prod_merc_positions_long             Prod_Merc_Positions_Long_All
PM Short    prod_merc_positions_short            Prod_Merc_Positions_Short_All
SD Long     swap_positions_long_all              Swap_Positions_Long_All
SD Short    swap__positions_short_all            Swap__Positions_Short_All
MM Long     m_money_positions_long_all           M_Money_Positions_Long_All
MM Short    m_money_positions_short_all          M_Money_Positions_Short_All
OR Long     other_rept_positions_long            Other_Rept_Positions_Long_All
OR Short    ot

In [32]:
cols = list(raw.columns)

# --- Group 1: Identifiers & metadata ---
id_cols = [c for c in cols if any(k in c.lower() for k in [
    'market', 'exchange', 'cftc', 'contract', 'date', 'futonly', 'code',
])]

# --- Group 2: Open Interest ---
oi_cols = [c for c in cols if 'open_interest' in c.lower() and c not in id_cols]

# --- Group 3: Position columns by trader category ---
trader_prefixes = {
    'Producer/Merchant (PM)': 'Prod_Merc',
    'Swap Dealers (SD)':      'Swap',
    'Managed Money (MM)':     'M_Money',
    'Other Reportables (OR)': 'Other_Rept',
    'Total Reportable':       'Tot_Rept',
    'Non-Reportable (NR)':    'NonRept',
}
trader_groups = {}
assigned = set(id_cols + oi_cols + ['date'])
for label, prefix in trader_prefixes.items():
    trader_groups[label] = [c for c in cols if c.startswith(prefix)]
    assigned.update(trader_groups[label])

# --- Group 4: Concentration ratios ---
conc_cols = [c for c in cols if 'conc' in c.lower() or 'Conc' in c]
assigned.update(conc_cols)

# --- Group 5: Number of traders ---
trader_count_cols = [c for c in cols if 'traders' in c.lower() or 'Traders' in c]
assigned.update(trader_count_cols)

# --- Group 6: Change columns ---
change_cols = [c for c in cols if 'change' in c.lower() or 'Change' in c]
assigned.update(change_cols)

# --- Group 7: Percent of OI columns ---
pct_cols = [c for c in cols if 'pct' in c.lower() or 'Pct' in c or 'percent' in c.lower()]
assigned.update(pct_cols)

# --- Unassigned ---
unassigned = [c for c in cols if c not in assigned]

# Print summary
print('='*80)
print('COLUMN GROUPS')
print('='*80)

print(f'\n--- Identifiers & Metadata ({len(id_cols)}) ---')
for c in id_cols:
    print(f'  {c}')

print(f'\n--- Open Interest ({len(oi_cols)}) ---')
for c in oi_cols:
    print(f'  {c}')

for label, group in trader_groups.items():
    print(f'\n--- {label} positions ({len(group)}) ---')
    for c in sorted(group):
        print(f'  {c}')

print(f'\n--- Concentration Ratios ({len(conc_cols)}) ---')
for c in conc_cols:
    print(f'  {c}')

print(f'\n--- Number of Traders ({len(trader_count_cols)}) ---')
for c in trader_count_cols:
    print(f'  {c}')

print(f'\n--- Weekly Changes ({len(change_cols)}) ---')
for c in change_cols:
    print(f'  {c}')

print(f'\n--- Percent of OI ({len(pct_cols)}) ---')
for c in pct_cols:
    print(f'  {c}')

if unassigned:
    print(f'\n--- Unassigned ({len(unassigned)}) ---')
    for c in unassigned:
        print(f'  {c}')

total = len(id_cols) + len(oi_cols) + sum(len(g) for g in trader_groups.values()) \
        + len(conc_cols) + len(trader_count_cols) + len(change_cols) + len(pct_cols) + len(unassigned)
print(f'\nTotal accounted: {total} / {len(cols)} columns')

COLUMN GROUPS

--- Identifiers & Metadata (14) ---
  Market_and_Exchange_Names
  As_of_Date_In_Form_YYMMDD
  As_of_Date_Form_MM/DD/YYYY
  CFTC_Contract_Market_Code
  CFTC_Market_Code
  CFTC_Region_Code
  CFTC_Commodity_Code
  Contract_Units
  CFTC_Contract_Market_Code_Quotes
  CFTC_Market_Code_Quotes
  CFTC_Commodity_Code_Quotes
  CFTC_SubGroup_Code
  FutOnly_or_Combined
  date

--- Open Interest (7) ---
  Open_Interest_All
  Open_Interest_Old
  Open_Interest_Other
  Change_in_Open_Interest_All
  Pct_of_Open_Interest_All
  Pct_of_Open_Interest_Old
  Pct_of_Open_Interest_Other

--- Producer/Merchant (PM) positions (6) ---
  Prod_Merc_Positions_Long_All
  Prod_Merc_Positions_Long_Old
  Prod_Merc_Positions_Long_Other
  Prod_Merc_Positions_Short_All
  Prod_Merc_Positions_Short_Old
  Prod_Merc_Positions_Short_Other

--- Swap Dealers (SD) positions (15) ---
  Swap_Positions_Long_All
  Swap_Positions_Long_Old
  Swap_Positions_Long_Other
  Swap_Positions_Short_All
  Swap_Positions_Short_Old
  